# Typed Fact Extraction Pipeline — Scalar Facts Only

Extracts **scalar attribute facts** (headcount, dates, money amounts, role titles)
as reified Fact nodes. Relations (usesTool, worksFor, hasPart, …) are **NOT** handled
here — they live as typed entity→entity edges in `relation_extraction.ipynb`.

**Why the split:**
- PPR walks the relation layer (`RELATED_TO`-family edges). `HAS_FACT` is invisible to PPR.
- If relations were stored as Fact nodes, they would be removed from the traversal graph,
  killing multi-hop connectivity ("who owns the tool team X uses").
- Scalar facts have no second entity on the far end, so they can't be edges — they need nodes.
- Relations have a second entity, so they ARE edges.

**Pipeline:**
1. **Stage 1** — LLM extracts scalar claims only; explicit prompt to skip entity→entity relations.
2. **Stage 2** — Deterministic normalizer (money → minor units, count, date → ISO, text verbatim).
3. **Stage 3** — Span re-verification against source quote.
4. **Stage 4** — Rule-based confidence (not logprob-based).

**Inputs:** `entity_cache.json`, `resolved_entities.json`
**Output:** `facts_cache.json` — scalar Fact records staged for Neo4j write

## Setup & imports

In [209]:
import json
import re
import sys
import hashlib
from decimal import Decimal, InvalidOperation
from enum import Enum
from pathlib import Path
from typing import Literal

import dateparser
from babel.numbers import get_currency_precision
from dotenv import load_dotenv
from openai import OpenAI
from price_parser import Price
from pydantic import BaseModel, Field, model_validator

load_dotenv()
client = OpenAI()

ENTITY_CACHE   = Path("entity_cache.json")
RESOLVED_CACHE = Path("resolved_entities.json")
FACTS_CACHE    = Path("facts_cache.json")

EXTRACTOR_VERSION = "fact-extraction-v0/gpt-4.1-mini"

for p in (ENTITY_CACHE, RESOLVED_CACHE):
    if not p.exists():
        sys.exit(f"missing cache: {p}")
print("caches found ✓")

caches found ✓


## Pydantic contract — Fact record

Single source of truth for the data shape.  
The LLM fills semantic slots and `raw_value_str`. Typed scalar fields are filled by the normalizer (Stage 2), never by the LLM.

In [211]:
from typing import NamedTuple

FACT_TYPE_VALUES = [
    "assertion",       # checkable standalone claim — the ONLY type stored as a Fact node
    "definition",      # explains what X is or means
    "opinion",         # normative/evaluative ("X is bad")
    "instruction",     # imperative or recommendation ("do X", "never Y")
    "metaphor",        # analogy or comparison ("X is like Y")
    "prediction",      # speculative claim about the future
    "other_non_fact",
]

VALUE_TYPE_VALUES = ["money", "count", "percentage", "date", "duration", "text"]


class FactRecord(BaseModel):
    """Scalar-only reified Fact node.

    Relational predicates (usesTool, ownerOf, hasPart, worksFor, partOf,
    documentedIn, responsibleFor, ...) are NOT facts — they are typed
    entity-to-entity edges in the relation layer that PPR traverses.
    Facts hold only scalar attribute values (money, count, date, text).
    """

    # identity
    fact_id: str = Field(description="stable hash of (doc_id, char_start, predicate, subject)")

    # semantic slots (LLM fills these)
    subject_entity_id: str = Field(description="canonical resolved entity ID")
    predicate:         str   # from the scalar predicate enum
    predicate_raw:     str   = Field(default="")
    value_type:        str   = "text"   # money | count | percentage | date | duration | text
    raw_value_str:     str   = Field(description="verbatim value copied from source")
    pred_set_name:     str   = "scalar-v1"

    # fact classification (Fix 1)
    fact_type: str = "assertion"

    # typed scalar values — Stage 2 fills at most ONE group per fact
    amount_minor:  int | None = None   # money: integer minor units (never float)
    currency:      str | None = None   # money: ISO 4217
    count:         int | None = None   # count
    approximate:   bool       = False  # count: was value prefixed with ~
    basis_points:  int | None = None   # percentage: exact integer
    iso_date:      str | None = None   # date: ISO 8601
    iso_duration:  str | None = None   # duration: ISO 8601
    text_value:    str | None = None   # text / other — no parse

    # bi-temporal validity
    valid_from:        str | None = None
    valid_to:          str | None = None
    temporal_inferred: bool       = False

    # provenance
    doc_id:       str
    char_start:   int
    char_end:     int
    source_quote: str

    # pipeline stamps
    span_verified:     bool  = False
    norm_ok:           bool  = False
    confidence:        float = 1.0
    extractor_version: str   = EXTRACTOR_VERSION


class ChunkResult(NamedTuple):
    """Return type from process_chunk."""
    facts:             list[FactRecord]
    non_facts:         list[dict]       # rejected by fact_type gate (Fix 1)
    rejected_subjects: list[dict]       # rejected by subject validation (Fix 2)


def make_fact_id(doc_id: str, char_start: int, predicate: str, subject: str) -> str:
    key = f"{doc_id}|{char_start}|{predicate}|{subject}"
    return hashlib.sha1(key.encode()).hexdigest()[:16]


print("Pydantic contract loaded (scalar-only) ✓")
print(f"  value_type values: {VALUE_TYPE_VALUES}")


Pydantic contract loaded (scalar-only) ✓
  value_type values: ['money', 'count', 'percentage', 'date', 'duration', 'text']


## Load caches

Builds two lookup structures the pipeline needs at every stage:
- `mention_to_canonical` — surface form → canonical entity ID
- `chunks` — deduplicated (episode, source_file) pairs with recovered char offsets

In [213]:
# ── resolved entities ─────────────────────────────────────────────────────────
resolved_data = json.loads(RESOLVED_CACHE.read_text())
mention_to_canonical: dict[str, str] = resolved_data["mention_to_canonical"]
print(f"resolved entities : {resolved_data['total_entities']:>5}")
print(f"surface forms     : {len(mention_to_canonical):>5}")

# ── entity cache — deduplicate chunks ────────────────────────────────────────
entity_cache = json.loads(ENTITY_CACHE.read_text())
mentions     = entity_cache["mentions"]

# One chunk = (source_file, episode_idx). Recover char offsets by searching
# the episode text in the source document. Cache doc text to avoid re-reading.
_doc_cache: dict[str, str] = {}

def _doc_text(source_file: str) -> str:
    if source_file not in _doc_cache:
        p = Path(source_file)
        _doc_cache[source_file] = p.read_text(encoding="utf-8", errors="ignore") if p.exists() else ""
    return _doc_cache[source_file]


# Deduplicate: one chunk entry per (source_file, episode_idx)
ChunkKey = tuple[str, str]  # (source_file, episode_idx)
chunks: dict[ChunkKey, dict] = {}

for m in mentions:
    key: ChunkKey = (m["source_file"], str(m["episode_idx"]))
    if key in chunks:
        continue
    episode = m["episode"]
    doc     = _doc_text(m["source_file"])
    offset  = doc.find(episode)
    chunks[key] = {
        "doc_id":     m["source_file"],
        "episode_idx": str(m["episode_idx"]),
        "text":       episode,
        "char_start": offset,
        "char_end":   offset + len(episode) if offset >= 0 else -1,
    }

# Collect known entities per chunk (surface forms that appear in the episode)
chunk_entities: dict[ChunkKey, list[str]] = {k: [] for k in chunks}
for m in mentions:
    key = (m["source_file"], str(m["episode_idx"]))
    canon = mention_to_canonical.get(m["text"])
    if canon and canon not in chunk_entities[key]:
        chunk_entities[key].append(canon)

bad_offsets = sum(1 for c in chunks.values() if c["char_start"] < 0)
print(f"\nchunks            : {len(chunks):>5}")
print(f"bad char offsets  : {bad_offsets:>5}  (source file missing/moved)")

resolved entities :  1026
surface forms     :   880

chunks            :   182
bad char offsets  :    33  (source file missing/moved)


## Stage 1 — Schema-guided LLM extraction (scalar facts only)

The model receives the chunk text + the list of canonical entity IDs present.
It emits scalar claim records: subject entity, scalar predicate, `raw_value_str` verbatim.

**Explicit instruction to skip relations**: "team X uses SharePoint" is a relation — skip it here.
**Fact gate**: labels each claim by type; non-assertions are routed to the non-fact log.
**Subject gate**: subject must come from the ENTITIES list — invalid subjects are rejected post-hoc.

In [217]:
from dataclasses import dataclass


@dataclass
class PredicateSet:
    """Holds one complete predicate vocabulary for Stage 1."""
    name:     str
    contract: dict[str, tuple[str, str]]   # pred -> (value_type, norm_handler)
    glosses:  str

    @property
    def enum_values(self) -> list[str]:
        return list(self.contract.keys())


FEW_SHOT_NEGATIVES = """EXAMPLES — what to reject:

SKIP (relation — goes to relation layer, not here):
  "team X uses SharePoint"          → usesTool relation, NOT a scalar fact
  "GraphRAG has component Y"        → hasPart relation, NOT a scalar fact
  "Alice works for Acme Corp"       → worksFor relation, NOT a scalar fact

REJECT (metaphor):
  "gold mining like where people are exploding dynamite"  → fact_type: metaphor

REJECT (opinion):
  "Customer conversations are bad by default"  → fact_type: opinion

REJECT (instruction):
  "never talking about the idea!"  → fact_type: instruction

REJECT (definition):
  "CAGR measures how fast something grows per year..."  → fact_type: definition

KEEP (scalar assertion):
  'the team has 12 people'          -> headcount = '12'
  'founded in 2019'                 -> foundingDate = '2019'
  'raised $5M in seed funding'      -> fundingRaised = '$5M'"""


def _build_system(pred_set: PredicateSet) -> str:
    return f"""You extract SCALAR ATTRIBUTE FACTS from a text chunk.

A scalar fact is a measured or stated value about ONE entity:
  headcount, a founding date, an amount raised, a role title, industry.

You do NOT extract relations between entities here.
  "team X uses SharePoint" is a relation (usesTool) — skip it.
  "team X has part Y" is a relation (hasPart) — skip it.
  "team X has 12 people" is a scalar fact (headcount=12) — keep it.

{FEW_SHOT_NEGATIVES}

FACT CLASSIFICATION — label every claim you find with fact_type:
  assertion     — checkable standalone scalar claim about a specific entity (KEEP)
  definition    — explains what X is (REJECT)
  opinion       — normative claim: "X is bad" (REJECT)
  instruction   — imperative: "do X", "never Y" (REJECT)
  metaphor      — analogy: "X is like Y" (REJECT)
  prediction    — speculative future claim (REJECT)
  other_non_fact — doesn't fit but is not an assertion (REJECT)

You will receive:
- TEXT: the source chunk
- ENTITIES: canonical entity IDs — the ONLY valid subjects

For each scalar claim (assertion OR labeled non-fact), emit a record:
- subject_entity_id: copy VERBATIM from the ENTITIES list
- predicate: from the closed scalar enum; use "other" if none fits
- predicate_raw: what you wanted (required when predicate="other")
- raw_value_str: copy the value VERBATIM from the source, character-for-character.
  NEVER compute or parse the value. "raised $5M" stays "$5M".
- valid_from / valid_to: ISO dates ONLY if explicitly stated; else null
- temporal_inferred: true if you inferred valid-time rather than reading it

Predicate glosses (scalar attributes only):
{pred_set.glosses}"""


def _build_schema(pred_set: PredicateSet) -> dict:
    return {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "facts": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "subject_entity_id": {"type": "string"},
                        "predicate":         {"type": "string", "enum": pred_set.enum_values},
                        "predicate_raw":     {"type": "string"},
                        "raw_value_str":     {"type": "string"},
                        "fact_type":         {"type": "string", "enum": FACT_TYPE_VALUES},
                        "valid_from":        {"type": ["string", "null"]},
                        "valid_to":          {"type": ["string", "null"]},
                        "temporal_inferred": {"type": "boolean"},
                    },
                    "required": [
                        "subject_entity_id", "predicate", "predicate_raw",
                        "raw_value_str", "fact_type",
                        "valid_from", "valid_to", "temporal_inferred",
                    ],
                },
            }
        },
        "required": ["facts"],
    }


EXTRACT_MODEL = "gpt-4.1-mini"


def extract_facts_from_chunk(
    chunk: dict, entities: list[str], pred_set: PredicateSet
) -> list[dict]:
    """Stage 1: LLM extracts and classifies scalar claims in one chunk."""
    if not entities:
        return []
    user_msg = (
        f"TEXT:\n{chunk['text']}\n\n"
        "ENTITIES:\n" + "\n".join(f"- {e}" for e in entities)
    )
    resp = client.chat.completions.create(
        model=EXTRACT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": _build_system(pred_set)},
            {"role": "user",   "content": user_msg},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "scalar_facts",
                "strict": True,
                "schema": _build_schema(pred_set),
            },
        },
    )
    return json.loads(resp.choices[0].message.content)["facts"]


# ── scalar-v1 predicate set ───────────────────────────────────────────────────
# Group 2: scalar numeric/date — these are the core Fact nodes
# Group 3: scalar categorical/text — kept as text Facts (bi-temporal titles/labels)
# REMOVED: usesTool, ownerOf, worksFor, documentedIn, partOf, responsibleFor
#          → now live in relation_extraction.ipynb as entity→entity edges

_SCALAR_CONTRACT: dict[str, tuple[str, str]] = {
    # group 2 — scalar numeric / date
    "headcount":     ("attribute", "count"),
    "foundingDate":  ("attribute", "date"),
    "revenue":       ("attribute", "money"),
    "fundingRaised": ("attribute", "money"),
    "changedOn":     ("attribute", "date"),
    # group 3 — scalar categorical / text (bi-temporal titles and labels)
    # NOTE: if these don't need history, promote to entity properties instead.
    "hasRole":       ("attribute", "text"),
    "industry":      ("attribute", "text"),
    "fundingStage":  ("attribute", "text"),
    # escape hatch
    "other":         ("attribute", "text"),
}

_SCALAR_GLOSSES = "\n".join([
    "headcount     [count]  — size of an org or team in people. Value: a number.",
    "foundingDate  [date]   — when an org was founded. Value: a date string.",
    "revenue       [money]  — org revenue. Value: an amount string like '$5M'.",
    "fundingRaised [money]  — amount an org has raised. Value: an amount string.",
    "changedOn     [date]   — a stated event or change date. Value: a date string.",
    "hasRole       [text]   — subject holds role or title X. Value: the title string.",
    "                         Role stays a string; do NOT emit worksFor relation here.",
    "industry      [text]   — org vertical (fintech, biotech, …). Value: the label.",
    "fundingStage  [text]   — seed / Series A / etc. Value: the stage label.",
    "other         [text]   — no scalar predicate fits; fill predicate_raw.",
    "                         Use 'other' rather than forcing a scalar onto a relation.",
])

PRED_SET_SCALAR = PredicateSet(
    name="scalar-v1",
    contract=_SCALAR_CONTRACT,
    glosses=_SCALAR_GLOSSES,
)

print("Stage 1 ready (scalar-only) ✓")
print(f"  {len(_SCALAR_CONTRACT)} scalar predicates  "
      f"(relational predicates removed — see relation_extraction.ipynb)")


Stage 1 ready (scalar-only) ✓
  9 scalar predicates  (relational predicates removed — see relation_extraction.ipynb)


## Stage 2 — Deterministic scalar normalizer

Parses `raw_value_str` into a typed scalar based on the predicate contract.
**If parsing fails, `norm_ok=False` — never guess.**

| norm_handler | output                            |
|---|---|
| `money`      | `amount_minor: int` + `currency` (ISO 4217, never float) |
| `count`      | `count: int`, `approximate: bool` |
| `percentage` | `basis_points: int`               |
| `date`       | `iso_date: str` (ISO 8601)        |
| `duration`   | `iso_duration: str` (ISO 8601)    |
| `text`       | `text_value: str` (verbatim)      |

In [221]:
_SYMBOL_TO_ISO: dict[str, str] = {
    "$": "USD", "\u20ac": "EUR", "\u00a3": "GBP", "\u00a5": "JPY",
    "\u20b9": "INR", "\u20a9": "KRW", "\u20aa": "ILS", "\u20ba": "TRY",
    "A$": "AUD", "C$": "CAD", "HK$": "HKD", "S$": "SGD",
}

_APPROX_RE = re.compile(r"^[~\u2248\u2264\u2265><circa ]*", re.IGNORECASE)
_COUNT_RE  = re.compile(r"[\d,]+")
_PCT_RE    = re.compile(r"([\d.]+)\s*%")


def _normalize_money(raw: str) -> dict:
    p = Price.fromstring(raw)
    if p.amount is None:
        return {"norm_ok": False}
    symbol = (p.currency or "").strip()
    iso    = _SYMBOL_TO_ISO.get(symbol, symbol.upper() or "USD")
    try:
        precision = get_currency_precision(iso)
    except Exception:
        precision = 2
    amount_minor = int(Decimal(str(p.amount)) * (10 ** precision))
    return {"norm_ok": True, "amount_minor": amount_minor, "currency": iso}


def _normalize_count(raw: str) -> dict:
    approx = bool(_APPROX_RE.match(raw).group(0).strip())
    digits = _COUNT_RE.search(raw.replace(",", ""))
    if not digits:
        return {"norm_ok": False}
    return {"norm_ok": True, "count": int(digits.group()), "approximate": approx}


def _normalize_percentage(raw: str) -> dict:
    m = _PCT_RE.search(raw)
    if not m:
        return {"norm_ok": False}
    try:
        bp = int(Decimal(m.group(1)) * 100)
        return {"norm_ok": True, "basis_points": bp}
    except InvalidOperation:
        return {"norm_ok": False}


def _normalize_date(raw: str) -> dict:
    dt = dateparser.parse(raw, settings={"RETURN_AS_TIMEZONE_AWARE": False})
    if dt is None:
        return {"norm_ok": False}
    return {"norm_ok": True, "iso_date": dt.date().isoformat()}


def _normalize_duration(raw: str) -> dict:
    return {"norm_ok": True, "iso_duration": raw}


def normalize(
    predicate: str,
    raw_value_str: str,
    pred_set: "PredicateSet | None" = None,
) -> dict:
    """Stage 2: dispatch to the right scalar parser based on contract."""
    contract = pred_set.contract if pred_set is not None else _SCALAR_CONTRACT
    _, norm_handler = contract.get(predicate, ("attribute", "text"))
    if norm_handler == "money":      return _normalize_money(raw_value_str)
    if norm_handler == "count":      return _normalize_count(raw_value_str)
    if norm_handler == "percentage": return _normalize_percentage(raw_value_str)
    if norm_handler == "date":       return _normalize_date(raw_value_str)
    if norm_handler == "duration":   return _normalize_duration(raw_value_str)
    return {"norm_ok": True, "text_value": raw_value_str}


def value_type_for(predicate: str, pred_set: "PredicateSet | None" = None) -> str:
    contract = pred_set.contract if pred_set is not None else _SCALAR_CONTRACT
    _, norm_handler = contract.get(predicate, ("attribute", "text"))
    return norm_handler  # money | count | percentage | date | duration | text


print("Stage 2 normalizer ready (scalar-only) ✓")


Stage 2 normalizer ready (scalar-only) ✓


## Stage 3 — Span re-verification

Deterministically confirms `raw_value_str` (or its digit sequence) appears inside `source_quote`.  
If absent the LLM likely paraphrased or hallucinated — `span_verified=False` drops confidence sharply.

Also checks temporal slots: if `valid_from`/`valid_to` don't appear in the source text, sets `temporal_inferred=True`.

In [224]:
_DIGITS_RE = re.compile(r"\d+")


def verify_span(raw_value_str: str, source_quote: str,
                valid_from: str | None, valid_to: str | None) -> dict:
    """Stage 3: check raw_value_str and temporal slots appear in source_quote."""
    quote_lower = source_quote.lower()
    raw_lower   = raw_value_str.lower().strip()

    # primary check: verbatim substring
    if raw_lower and raw_lower in quote_lower:
        span_verified = True
    else:
        # fallback: all digit sequences from raw appear in quote
        digits = _DIGITS_RE.findall(raw_value_str)
        span_verified = bool(digits) and all(d in source_quote for d in digits)

    # temporal check: if a date was stated, its digits should be in the source
    temporal_inferred = False
    for date_val in filter(None, [valid_from, valid_to]):
        date_digits = _DIGITS_RE.findall(date_val)
        if date_digits and not all(d in source_quote for d in date_digits):
            temporal_inferred = True
            break

    return {"span_verified": span_verified, "temporal_inferred": temporal_inferred}


print("Stage 3 verifier ready ✓")

Stage 3 verifier ready ✓


## Stage 4 — Confidence scoring

Rule-based composite. **Not** logprob-based — documented heuristic.

```
start                1.00
norm_ok=False       -0.25   value couldn't be parsed into a typed scalar
span_verified=False -0.40   value not found verbatim in source (highest risk)
temporal_inferred   -0.10   valid-time was inferred, not stated
predicate=other     -0.10   no named predicate fit
object_is_clause    -0.20   relational object looks like prose, not an entity
```
Floor: 0.0.
If the distribution is still flat after weighting, report that — don't hide it.

In [227]:
def score_confidence(
    norm_ok: bool,
    span_verified: bool,
    temporal_inferred: bool,
    predicate: str,
) -> float:
    """Stage 4: rule-based confidence for scalar facts. NOT logprob-based.

    Formula:
      start                1.00
      norm_ok=False       -0.25   value could not be parsed (e.g. entity name as a count)
      span_verified=False -0.40   value not found verbatim in source
      temporal_inferred   -0.10   valid-time inferred, not stated
      predicate=other     -0.10   no named predicate fit

    If the distribution is flat after applying these, report it; don't hide it.
    """
    score = 1.00
    if not norm_ok:          score -= 0.25
    if not span_verified:    score -= 0.40
    if temporal_inferred:    score -= 0.10
    if predicate == "other": score -= 0.10
    return round(max(0.0, score), 2)


print("Stage 4 confidence scorer ready ✓")
print("  formula: 1.0 - 0.25(norm_fail) - 0.40(span_fail) - 0.10(temporal) - 0.10(other)")


Stage 4 confidence scorer ready ✓
  formula: 1.0 - 0.25(norm_fail) - 0.40(span_fail) - 0.10(temporal) - 0.10(other)


## Full pipeline — assemble one FactRecord per LLM extraction

In [230]:
def process_chunk(
    chunk: dict,
    entities: list[str],
    pred_set: "PredicateSet | None" = None,
) -> ChunkResult:
    """Run all four stages on one chunk. Returns scalar FactRecords only.

    Relational predicates are NOT handled here — see relation_extraction.ipynb.
    """
    ps = pred_set if pred_set is not None else PRED_SET_SCALAR
    raw_items = extract_facts_from_chunk(chunk, entities, ps)

    facts: list[FactRecord] = []
    non_facts: list[dict] = []
    rejected_subjects: list[dict] = []

    entity_set = set(entities)

    for rf in raw_items:
        fact_type = rf.get("fact_type", "assertion")
        pred      = rf["predicate"]
        raw_val   = rf["raw_value_str"]

        # Fix 1: gate non-assertions into non_fact bucket ─────────────────────
        if fact_type != "assertion":
            non_facts.append({
                "fact_type":         fact_type,
                "subject_entity_id": rf["subject_entity_id"],
                "predicate_raw":     rf.get("predicate_raw", pred),
                "raw_value_str":     raw_val,
                "doc_id":            chunk["doc_id"],
                "source_quote":      chunk["text"][:120],
            })
            continue

        # Fix 2: subject must be in the canonical entity list ──────────────────
        subj = rf["subject_entity_id"]
        if subj not in entity_set:
            rejected_subjects.append({
                "subject_attempted": subj,
                "predicate":         pred,
                "raw_value_str":     raw_val,
                "doc_id":            chunk["doc_id"],
                "note":              "not in canonical entity list — possible recall miss",
            })
            continue

        source_quote = chunk["text"]
        char_start   = chunk["char_start"]
        char_end     = chunk["char_end"]

        norm   = normalize(pred, raw_val, ps)
        verify = verify_span(raw_val, source_quote,
                             rf.get("valid_from"), rf.get("valid_to"))
        conf   = score_confidence(
            norm_ok          = norm.get("norm_ok", False),
            span_verified    = verify["span_verified"],
            temporal_inferred= verify["temporal_inferred"] or rf.get("temporal_inferred", False),
            predicate        = pred,
        )

        record = FactRecord(
            fact_id          = make_fact_id(chunk["doc_id"], char_start, pred, subj),
            subject_entity_id= subj,
            predicate        = pred,
            predicate_raw    = rf.get("predicate_raw", ""),
            value_type       = value_type_for(pred, ps),
            raw_value_str    = raw_val,
            pred_set_name    = ps.name,
            fact_type        = fact_type,
            valid_from       = rf.get("valid_from"),
            valid_to         = rf.get("valid_to"),
            temporal_inferred= verify["temporal_inferred"] or rf.get("temporal_inferred", False),
            doc_id           = chunk["doc_id"],
            char_start       = char_start,
            char_end         = char_end,
            source_quote     = source_quote,
            span_verified    = verify["span_verified"],
            norm_ok          = norm.get("norm_ok", False),
            confidence       = conf,
            **{k: v for k, v in norm.items() if k != "norm_ok"},
        )
        facts.append(record)

    return ChunkResult(facts=facts, non_facts=non_facts, rejected_subjects=rejected_subjects)


print("pipeline assembled (scalar-only) ✓")


pipeline assembled (scalar-only) ✓


## Demo run — 5 chunks

Picks the first 5 chunks that have at least one known entity. Prints typed Fact records with confidence and verification flags.  
Do **not** run the full corpus here — use the bulk cell below.

In [233]:
from collections import Counter

DEMO_CHUNKS = 10

demo_facts:     list[FactRecord] = []
demo_non_facts: list[dict]       = []
demo_rejected:  list[dict]       = []

chunk_items = [
    (key, chunk) for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
][:DEMO_CHUNKS]

for key, chunk in chunk_items:
    entities = chunk_entities[key]
    try:
        result = process_chunk(chunk, entities, PRED_SET_SCALAR)
        demo_facts.extend(result.facts)
        demo_non_facts.extend(result.non_facts)
        demo_rejected.extend(result.rejected_subjects)
    except Exception as e:
        print(f"  chunk {key} failed: {e}")

total_emitted = len(demo_facts) + len(demo_non_facts) + len(demo_rejected)
print("=" * 60)
print(f"  SCALAR FACT EXTRACTION REPORT  ({DEMO_CHUNKS} chunks)")
print("=" * 60)

# Gate stats
print(f"  total LLM claims      : {total_emitted}")
print(f"  assertions (scalar)   : {len(demo_facts)}")
print(f"  non-facts rejected    : {len(demo_non_facts)}")
for ft, cnt in sorted(Counter(n["fact_type"] for n in demo_non_facts).items(), key=lambda x: -x[1]):
    print(f"    {ft:<20} {cnt}")
print(f"  invalid subjects      : {len(demo_rejected)}")
print()

# Confidence distribution
if demo_facts:
    confs = sorted(r.confidence for r in demo_facts)
    buckets = Counter(round(c, 1) for c in confs)
    print(f"  CONFIDENCE  min={confs[0]:.2f} max={confs[-1]:.2f} "
          f"mean={sum(confs)/len(confs):.2f} spread={confs[-1]-confs[0]:.2f}")
    for score in sorted(buckets):
        print(f"    {score:.1f}  {'█' * buckets[score]} ({buckets[score]})")
    print()

# Surviving scalar facts
print(f"  SCALAR FACTS ({len(demo_facts)})")
print("-" * 60)
for r in sorted(demo_facts, key=lambda x: -x.confidence):
    ok  = "✓" if r.span_verified else "✗"
    nrm = "✓" if r.norm_ok else "✗"
    print(f"  [{r.confidence:.2f}] span={ok} norm={nrm} [{r.value_type}]")
    print(f"    subject : {r.subject_entity_id}")
    print(f"    pred    : {r.predicate}" + (f" ({r.predicate_raw})" if r.predicate_raw else ""))
    print(f"    raw     : {r.raw_value_str!r}")
    if r.amount_minor is not None:
        print(f"    typed   : {r.amount_minor} {r.currency} (minor-units)")
    elif r.count is not None:
        print(f"    typed   : {'~' if r.approximate else ''}{r.count}")
    elif r.iso_date:
        print(f"    typed   : {r.iso_date}")
    elif r.text_value:
        print(f"    typed   : {r.text_value!r}")
    print(f"    source  : {Path(r.doc_id).name}  [{r.char_start}:{r.char_end}]")
    print()

if demo_non_facts:
    print(f"  NON-FACT LOG ({len(demo_non_facts)})")
    print("-" * 60)
    for n in demo_non_facts:
        print(f"  [{n['fact_type']:<16}] {n['raw_value_str'][:70]!r}")


  SCALAR FACT EXTRACTION REPORT  (10 chunks)
  total LLM claims      : 16
  assertions (scalar)   : 16
  non-facts rejected    : 0
  invalid subjects      : 0

  CONFIDENCE  min=0.90 max=1.00 mean=0.91 spread=0.10
    0.9  ███████████████ (15)
    1.0  █ (1)

  SCALAR FACTS (16)
------------------------------------------------------------
  [1.00] span=✓ norm=✓ [text]
    subject : EntraID
    pred    : industry (identity and access management)
    raw     : 'identity and access management'
    typed   : 'identity and access management'
    source  : Microsoft Planet.md  [0:1085]

  [0.90] span=✓ norm=✓ [text]
    subject : Managed Identity
    pred    : other (Token-based access)
    raw     : 'Token-based access'
    typed   : 'Token-based access'
    source  : Microsoft Planet.md  [0:1085]

  [0.90] span=✓ norm=✓ [text]
    subject : Azure RBAC
    pred    : other (auth for Azure OpenAI model)
    raw     : '(maybe) auth for Azure OpenAI model'
    typed   : '(maybe) auth for Azure 

## Wikidata scalars added to scalar-v1

The Wikidata predicate experiment (from the previous run) mixed relational and
scalar predicates. The relational ones (`hasPart`, `uses`, `produces`, etc.) are
now in `relation_extraction.ipynb`. The Wikidata scalars that remain relevant here:

| predicate     | Wikidata PID | handler |
|---------------|--------------|---------|
| `employees`   | P1128 ✓      | count   |
| `inception`   | P571  ✓      | date    |
| `totalRevenue`| P2139 ✓      | money   |

These are added to `PRED_SET_SCALAR_WD` below if you want to test them.
Run the comparison cell to see them alongside `scalar-v1`.

In [235]:
# Wikidata scalar additions — a small extension of PRED_SET_SCALAR
_SCALAR_WD_CONTRACT: dict[str, tuple[str, str]] = {
    **_SCALAR_CONTRACT,
    "employees":   ("attribute", "count"),    # P1128 ✓ (alternative to headcount)
    "inception":   ("attribute", "date"),     # P571  ✓ (alternative to foundingDate)
    "totalRevenue":("attribute", "money"),    # P2139 ✓ (alternative to revenue)
}

_SCALAR_WD_GLOSSES = _SCALAR_GLOSSES + "\n".join([
    "",
    "employees    [count]  — headcount (Wikidata P1128). Use instead of headcount if preferred.",
    "inception    [date]   — founding/creation date (Wikidata P571).",
    "totalRevenue [money]  — org revenue (Wikidata P2139).",
])

PRED_SET_SCALAR_WD = PredicateSet(
    name="scalar-wd-v1",
    contract=_SCALAR_WD_CONTRACT,
    glosses=_SCALAR_WD_GLOSSES,
)
print(f"Wikidata scalar extension: {len(_SCALAR_WD_CONTRACT)} predicates")
print("  Relational Wikidata predicates (hasPart, uses, produces, ...) -> relation_extraction.ipynb")


Wikidata scalar extension: 12 predicates
  Relational Wikidata predicates (hasPart, uses, produces, ...) -> relation_extraction.ipynb


## Comparison run — same chunks, both predicate sets

In [200]:
# Compare scalar-v1 vs scalar-wd-v1 on the same chunks.
# The Wikidata relational predicates are NOT here anymore.
COMP_N = 10

comp_keys = [
    key for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
][:COMP_N]

comp_v0: dict[tuple, ChunkResult] = {}
comp_wd: dict[tuple, ChunkResult] = {}

for key in comp_keys:
    chunk = chunks[key]
    ents  = chunk_entities[key]
    try:
        comp_v0[key] = process_chunk(chunk, ents, PRED_SET_SCALAR)
        comp_wd[key] = process_chunk(chunk, ents, PRED_SET_SCALAR_WD)
    except Exception as e:
        print(f"  chunk {key} failed: {e}")

n_v0 = sum(len(v.facts) for v in comp_v0.values())
n_wd = sum(len(v.facts) for v in comp_wd.values())
print(f"scalar-v1:    {n_v0} facts")
print(f"scalar-wd-v1: {n_wd} facts")


scalar-v1:    20 facts
scalar-wd-v1: 12 facts


## Comparison report

In [202]:
from collections import Counter

def _hist(records: list[FactRecord]) -> Counter:
    return Counter(
        r.predicate + (f" ({r.predicate_raw})" if r.predicate_raw else "")
        for r in records
    )

def _other_pct(records: list[FactRecord]) -> str:
    if not records: return "n/a"
    n = sum(1 for r in records if r.predicate == "other")
    return f"{n}/{len(records)} = {100*n//len(records)}%"


all_v0 = [r for res in comp_v0.values() for r in res.facts]
all_wd = [r for res in comp_wd.values() for r in res.facts]

hist_v0 = _hist(all_v0)
hist_wd = _hist(all_wd)

# ── 1. Predicate histogram ─────────────────────────────────────────────────
print("=" * 60)
print(f"  PREDICATE HISTOGRAM  ({COMP_N} chunks, assertions only)")
print("=" * 60)

all_preds = sorted(
    set(hist_v0) | set(hist_wd),
    key=lambda p: -(hist_v0.get(p, 0) + hist_wd.get(p, 0)),
)
for p in all_preds:
    n_v = hist_v0.get(p, 0)
    n_w = hist_wd.get(p, 0)
    print(f"  {p[:32]:<32}  v0={n_v:>3}  wd={n_w:>3}  {'|' * max(n_v, n_w)}")

print()
print(f"  other rate -- v0: {_other_pct(all_v0)}   wd: {_other_pct(all_wd)}")
print()

# ── 2. Construction.md ────────────────────────────────────────────────────
print("=" * 60)
print("  CONSTRUCTION.md -- motivating case")
print("=" * 60)

def _print_chunk_facts(records: list[FactRecord], label: str) -> None:
    print(f"  [{label}]")
    if not records:
        print("    (none)")
        return
    for r in records:
        raw = r.predicate + (f" ({r.predicate_raw})" if r.predicate_raw else "")
        cla = " [clause]" if r.object_is_clause else ""
        print(f"    [{r.confidence:.2f}] {r.subject_entity_id[:20]:<20} --{raw}--> {r.raw_value_str[:40]!r}{cla}")

const_v0 = [r for r in all_v0 if "Construction" in r.doc_id]
const_wd = [r for r in all_wd if "Construction" in r.doc_id]
_print_chunk_facts(const_v0, "v0")
print()
_print_chunk_facts(const_wd, "wikidata-v1")
print()

# ── 3. Still other under BOTH sets ────────────────────────────────────────
print("=" * 60)
print("  STILL other IN BOTH (true coverage gaps)")
print("=" * 60)

other_v0 = {(r.subject_entity_id, r.raw_value_str) for r in all_v0 if r.predicate == "other"}
other_wd = {(r.subject_entity_id, r.raw_value_str) for r in all_wd if r.predicate == "other"}
both_other = other_v0 & other_wd

if not both_other:
    print("  (none)")
else:
    for subj, raw in sorted(both_other):
        print(f"  {subj[:30]:<30}  {raw[:55]!r}")
print()

# ── 4. Predicate migrations ────────────────────────────────────────────────
print("=" * 60)
print("  PREDICATE MIGRATIONS (same subject+raw, different predicate)")
print("=" * 60)

v0_idx = {(r.subject_entity_id, r.raw_value_str): r.predicate for r in all_v0}
wd_idx = {(r.subject_entity_id, r.raw_value_str): r.predicate for r in all_wd}

migrations = [
    (k, v0_idx[k], wd_idx[k])
    for k in set(v0_idx) & set(wd_idx)
    if v0_idx[k] != wd_idx[k]
]

if not migrations:
    print("  (no exact overlap with different predicate)")
else:
    for (subj, raw), p_v0, p_wd in sorted(migrations):
        print(f"  {subj[:24]:<24}  {p_v0:<20} -> {p_wd}")
        print(f"  {'':<24}  {raw[:55]!r}")
        print()


  PREDICATE HISTOGRAM  (10 chunks, assertions only)
  other (can communicate with each  v0=  4  wd=  0  ||||
  other (function)                  v0=  0  wd=  4  ||||
  other (Management / Observabilit  v0=  3  wd=  0  |||
  other (purpose)                   v0=  0  wd=  3  |||
  other (price)                     v0=  1  wd=  1  |
  other (more setup and more contr  v0=  1  wd=  0  |
  other (rental duration)           v0=  0  wd=  1  |
  other (storage of unavoidable se  v0=  1  wd=  0  |
  other (on Container Apps)         v0=  1  wd=  0  |
  other (for files)                 v0=  1  wd=  0  |
  other (deployment)                v0=  0  wd=  1  |
  other (location)                  v0=  0  wd=  1  |
  other (duration)                  v0=  1  wd=  0  |
  other (usage)                     v0=  0  wd=  1  |
  industry (identity and access ma  v0=  1  wd=  0  |
  other (inside Azure AI Foundry)   v0=  1  wd=  0  |
  other (accessed through SELECTED  v0=  1  wd=  0  |
  other (auth for Az

## Bulk run — all chunks → `facts_cache.json`

Processes every chunk that has at least one known entity and a valid char offset.  
Results are saved incrementally every 50 chunks.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm as tqdm_nb

BULK_WORKERS = 8
SAVE_EVERY   = 50

eligible = [
    (key, chunk) for key, chunk in chunks.items()
    if chunk["char_start"] >= 0 and chunk_entities.get(key)
]
print(f"{len(eligible)} eligible chunks")

all_facts:     list[dict] = []
all_non_facts: list[dict] = []

def _process(args: tuple) -> tuple[list[dict], list[dict]]:
    key, chunk = args
    result = process_chunk(chunk, chunk_entities[key], PRED_SET_SCALAR)
    return [r.model_dump(mode="json") for r in result.facts], result.non_facts

def _save() -> None:
    FACTS_CACHE.write_text(json.dumps(all_facts, ensure_ascii=False, indent=2))

with ThreadPoolExecutor(max_workers=BULK_WORKERS) as executor:
    futures = {executor.submit(_process, item): item for item in eligible}
    with tqdm_nb(total=len(eligible), desc="scalar facts") as bar:
        for i, future in enumerate(as_completed(futures)):
            fb, nfb = future.result()
            all_facts.extend(fb)
            all_non_facts.extend(nfb)
            bar.update(1)
            if (i + 1) % SAVE_EVERY == 0:
                _save()

_save()

conf_vals = [f["confidence"] for f in all_facts]
by_pred: dict[str, int] = {}
for f in all_facts:
    by_pred[f["predicate"]] = by_pred.get(f["predicate"], 0) + 1
non_fact_types: dict[str, int] = {}
for n in all_non_facts:
    non_fact_types[n["fact_type"]] = non_fact_types.get(n["fact_type"], 0) + 1

print(f"\n{len(all_facts)} scalar facts  +  {len(all_non_facts)} non-facts logged")
print(f"  avg confidence : {sum(conf_vals)/max(1,len(conf_vals)):.2f}")
print("\n  by predicate:")
for pred, cnt in sorted(by_pred.items(), key=lambda x: -x[1]):
    print(f"    {pred:<15} {cnt}")
print("\n  non-fact types:")
for ft, cnt in sorted(non_fact_types.items(), key=lambda x: -x[1]):
    print(f"    {ft:<20} {cnt}")


## Neo4j write — next step (not yet implemented)

`facts_cache.json` is the staging area. To write to Neo4j you need:
1. `NEO4J_PASSWORD` in `.env`
2. A Fact node schema / constraints defined in the database
3. A Cypher MERGE pattern that keys on `fact_id` for idempotency

The `Neo4jGraphDriver` in `services/neo4j.py` provides the async connection.  
Implement this as a separate cell or script once the schema is defined.